# Krediitkaardi pettuste tuvastamine

> **Märkus:** Jooksuta lahtreid **järjestikku ülevalt alla** (Shift+Enter). Kui jooksutad lahtreid valest järjekorrast, võivad tekkida vead nagu . Alusta alati ülevalt.

In [ ]:
# ── Teekide paigaldamine (jooksuta üks kord) ──────────────────────────────
%pip install imbalanced-learn lightgbm fpdf2 -q

# ── Põhilised impordid ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

print("Kõik teegid laaditud.")

## Probleemi püstitus

### Mida püüame ennustada?

Selles projektis uurime, kas on võimalik tuvastada **pettuslikke krediitkaardi tehinguid** automaatselt.

Mudel vaatab iga tehingut ja vastab küsimusele: **kas see tehing on pettus või normaalne?**

- **Sisend:** tehingu andmed (aeg, summa ja 28 anonüümitud tunnust)
- **Väljund:** 0 = normaalne tehing, 1 = pettus

Andmestik: [Credit Card Fraud Detection – Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

### Miks see probleem on oluline?

Krediitkaardipettused põhjustavad pangandussektoris igal aastal miljardeid eurosid kahju. Automaatne tuvastamine on vajalik, et:
- kaitsta kliente — nad ei peaks maksma ostude eest, mida nad ei teinud
- reageerida kiiresti — inimene ei jõua käsitsi kõiki tehinguid kontrollida
- vähendada kahju pankadele ja kaupmeestele

Andmestik on **tugevalt tasakaalustamata** — ainult 0,172% tehingutest on pettused. See tähendab, et lihtsad lahendused ei tööta hästi ja probleem on tehniliselt huvitav.

### Võimalikud uurimissuunad

- Kui hästi suudab mudel pettusi tuvastada?
- Millised tunnused on kõige informatiivsemad pettuse tuvastamisel?
- Kuidas mõjutab tasakaalustamata andmestik mudeli tulemust ja milliseid meetodeid (nt SMOTE) võiks selle leevendamiseks kasutada?

## Andmestiku kirjeldus

## Andmete allikas

**Allikas:** [Credit Card Fraud Detection – Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

### Kontekst
Krediitkaardi ettevõtetel on oluline tuvastada pettuslikud tehingud, et kliendid ei peaks maksma kaupade eest, mida nad ei ostnud.

### Sisu
Andmestik sisaldab Euroopa krediitkaardi omanike tehinguid septembrist 2013. Andmestik hõlmab kahe päeva tehinguid, milles on **492 pettust 284 807 tehingust**. Andmestik on tugevalt tasakaalustamata — positiivne klass (pettused) moodustab vaid **0,172%** kõigist tehingutest.

Andmestik sisaldab ainult arvulisi tunnuseid, mis on **PCA** transformatsiooni tulemus.

> **Mis on PCA?** PCA (Principal Component Analysis) on matemaatiline meetod andmete anonümiseerimiseks ja kokku pakkimiseks. Pank ei saanud avaldada päris tehinguandmeid (kaardi number, kauplus, asukoht) — see oleks konfidentsiaalne. PCA abil teisendati originaalandmed anonüümseteks tunnusteks V1–V28. Algandmeid ei saa tagasi arvutada, aga mustrid (sh pettuse mustrid) säilivad.

Konfidentsiaalsuse tõttu ei ole algandmed avalikustatud. Tunnused V1–V28 on PCA peakomponendid. Tunnused **Time** ja **Amount** ei ole PCA-ga teisendatud.

Klasside tasakaalustamatuse tõttu on soovitatav täpsuse mõõtmiseks kasutada **AUPRC-d** (Area Under the Precision-Recall Curve), mitte tavalise klassifitseerimistäpsuse meetrikat.

### Andmete laadimine

In [ ]:
import pandas as pd

df = pd.read_csv("creditcard.csv")
df.head()

## Tunnuste (feature'ite) kirjeldus

| Tunnus | Kirjeldus |
|--------|-----------|
| **Time** | Mitu sekundit on möödunud andmestiku esimesest tehingust |
| **V1–V28** | Anonüümitud tunnused — algandmed on konfidentsiaalsuse tõttu peidetud matemaatilise teisenduse taha. Me ei tea, mida need täpselt mõõdavad. |
| **Amount** | Tehingu summa eurodes |
| **Class** | 1 = pettus, 0 = normaalne tehing |

## Andmestiku suurus ja struktuur

In [ ]:
print("Ridade ja veergude arv:", df.shape)
print("\nVeergude nimed:")
print(df.columns.tolist())
print("\nPuuduvad väärtused:")
print(df.isnull().sum().sum(), "puuduvat väärtust")
print("\nAndmestiku info:")
df.info()

## Andmete analüüs (EDA)

## Põhistatistika (keskmine, jaotus jne)

In [ ]:
import matplotlib.pyplot as plt

# Statistiline kokkuvõte
print("Statistiline kokkuvõte:")
display(df.describe())

# Klasside jaotus (pettus vs. normaalne)
print("\nKlasside jaotus:")
print(df['Class'].value_counts())
print(f"\nPettuste osakaal: {df['Class'].mean()*100:.4f}%")

# Tehingusumma jaotus
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Amount'].hist(bins=50, ax=axes[0])
axes[0].set_title('Tehingusumma jaotus')
axes[0].set_xlabel('Summa')
axes[0].set_ylabel('Sagedus')

df['Class'].value_counts().plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'])
axes[1].set_title('Klasside jaotus')
axes[1].set_xlabel('Klass (0=normaalne, 1=pettus)')
axes[1].set_ylabel('Arv')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

**Mida graafikud näitavad?**

- **Tehingusumma jaotus (vasak):** Enamik tehinguid on väikese summaga (0–500€). Üle 5000€ tehinguid on väga vähe — igapäevased ostud on tavaliselt odavad.

- **Klasside jaotus (parem):** Normaalseid tehinguid on ~280 000, pettusi ainult 492. Punane tulp on nii väike, et seda peaaegu ei näe. See on meie suurim väljakutse — mudel peab leidma "nõela heinakuhjas".

## Visualiseeringud (graafikud)

In [ ]:
import seaborn as sns

# Tehingusumma jaotus klassi järgi
plt.figure(figsize=(10, 4))
sns.boxplot(x='Class', y='Amount', data=df)
plt.title('Tehingusumma jaotus klassi järgi')
plt.xlabel('Klass (0=normaalne, 1=pettus)')
plt.ylabel('Summa')
plt.tight_layout()
plt.show()

# Tehingusumma log-skaalal klassi järgi
plt.figure(figsize=(10, 4))
sns.boxplot(x='Class', y='Amount', data=df)
plt.yscale('log')
plt.title('Tehingusumma jaotus klassi järgi (log-skaala)')
plt.xlabel('Klass (0=normaalne, 1=pettus)')
plt.ylabel('Summa (log)')
plt.tight_layout()
plt.show()

# Ajaline jaotus klassi järgi
plt.figure(figsize=(10, 4))
sns.histplot(data=df, x='Time', hue='Class', bins=50, kde=True)
plt.title('Tehingute ajaline jaotus klassi järgi')
plt.xlabel('Aeg (sekundites)')
plt.ylabel('Sagedus')
plt.tight_layout()
plt.show()

**Mida see graafik näitab?**

- **Normaalsed tehingud (vasak, klass 0):** Enamik on väikese summaga, kuid esineb ka suuri tehinguid kuni 25 000€ — inimesed teevad nii väikesi kui suuri oste.
- **Pettused (parem, klass 1):** Pettused on peaaegu kõik väikese summaga, maksimaalselt ~2500€. Üle 5000€ pettusi praktiliselt pole.

**Miks?** Pettuse tegijad eelistavad väikseid summasid — suured tehingud tõmbavad rohkem tähelepanu ja aktiveerivad kiiremini kaardi blokeerimise. See on mudelile kasulik signaal.

**Log-skaala graafiku lugemine:**

**Log-skaala** tähendab, et y-telg ei kasva ühtlaselt (1, 2, 3...) vaid kümnekordselt (1 → 10 → 100 → 1000...). See aitab näha väikeste väärtuste erinevusi paremini, kui andmed on väga eri skaaladel.

- **Normaalsed tehingud (0):** mediaan ~20€, enamik tehinguid vahemikus 5–80€
- **Pettused (1):** mediaan ~10€, enamik tehinguid vahemikus 1–100€

**Tähtis järeldus:** Pettuste summad on küll veidi väiksemad, aga erinevus pole tegelikult nii suur. See tähendab, et **summa üksi ei ole piisav pettuse tuvastamiseks** — mudel peab toetuma rohkem V1–V28 tunnustele.

**Tehingute ajalise jaotuse lugemine:**

X-telg on aeg sekundites — andmestik katab 2 päeva (~172 800 sekundit).

- **Normaalsed tehingud (sinine):** Kaks selget tippu — üks ~75 000 ja teine ~140 000 sekundi juures. See vastab kahele päevale — hommikust õhtuni on tehinguid palju, öösel vähe (langus keskel ~100 000 juures).
- **Pettused (oranž):** Nii õhuke joon, et peaaegu ei näe — absoluutarvudes tundub ühtlane.

**NB:** See graafik näitab absoluutarve — kuna pettusi on väga vähe, ei ole trend nähtav. Osakaaluna vaadates (vt järgmine graafik) selgub, et teatud kellaaegadel on pettusi kuni 6–7 korda rohkem kui teistel.

### Pettuste osakaal ajas

Eelmine graafik näitas absoluutarve — pettuste joon oli nii madal, et trendi polnud näha. Vaatame hoopis **osakaalu**: mitu protsenti tehingutest igas ajavahemikus on pettused?

In [ ]:
import numpy as np

# Kasuta ajutist koopiat et df ei muutuks
df_time = df.copy()
bins = np.linspace(df_time['Time'].min(), df_time['Time'].max(), 25)
df_time['time_bin'] = pd.cut(df_time['Time'], bins=bins)

osakaal = df_time.groupby('time_bin')['Class'].mean() * 100

plt.figure(figsize=(12, 4))
osakaal.plot(kind='bar', color='tomato')
plt.title('Pettuste osakaal (%) erinevatel kellaaegadel')
plt.xlabel('Ajavahemik')
plt.ylabel('Pettuste osakaal (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Kõrgeim pettuste osakaal: {osakaal.max():.2f}% (vahemik: {osakaal.idxmax()})")
print(f"Madalaim pettuste osakaal: {osakaal.min():.2f}% (vahemik: {osakaal.idxmin()})")

**Pettuste osakaal ajas — kuidas lugeda:**

Iga tulp näitab: mitu protsenti selles ajavahemikus tehtud tehingutest olid pettused.

- **Kõrgeim osakaal ~1.17%** — umbes esimese päeva lõpus/öösel (~93 000–100 000 sekundit)
- **Madalaim osakaal ~0.04%** — teatud päevasel ajal (~115 000–122 000 sekundit)

**Tähtis järeldus:** Erinevus on **~6–7 korda** — teatud kellaaegadel on pettusi suhteliselt palju rohkem. See tähendab, et **aeg on siiski kasulik signaal** mudelile, isegi kui absoluutarvudes seda ei paistnud.

## Seoste leidmine tunnuste vahel

In [ ]:
# Korrelatsioon klassiga - tulpdiagramm
corr_with_class = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)

print("Top 10 tunnust, mis on klassiga kõige rohkem korreleeritud:")
print(corr_with_class.head(10))

plt.figure(figsize=(10, 5))
corr_with_class.head(10).plot(kind='bar', color='steelblue')
plt.title('Top 10 tunnuse korrelatsioon klassiga (pettus)')
plt.xlabel('Tunnus')
plt.ylabel('Korrelatsioonikordaja')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

# Korrelatsioon klassiga - heatmap (ainult klassiga seotud tunnused)
plt.figure(figsize=(10, 2))
sns.heatmap(corr_with_class.head(10).to_frame().T, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', linewidths=0.5)
plt.title('Top 10 tunnuse korrelatsioon klassiga')
plt.tight_layout()
plt.show()

# Paarisdiagramm - tasakaalustatud valim (võrdne arv pettusi ja normaalseid)
fraud = df[df['Class'] == 1]
normal = df[df['Class'] == 0].sample(len(fraud), random_state=42)
balanced_df = pd.concat([fraud, normal])

top_features = corr_with_class.head(5).index.tolist() + ['Class']
sns.pairplot(balanced_df[top_features], hue='Class', plot_kws={'alpha': 0.5})
plt.suptitle('Paarisdiagramm top 5 tunnuse vahel (tasakaalustatud valim)', y=1.02)
plt.show()

**Graafiku lugemine:**

- **Negatiivne korrelatsioon (tulbad allpool 0):** V17, V14, V12 jt — kui nende väärtus on madal, on suurem tõenäosus, et tehing on pettus.
- **Positiivne korrelatsioon (tulbad ülespool 0):** V11, V4 — kui nende väärtus on kõrge, on suurem tõenäosus, et tehing on pettus.
- **Tugevaim signaal on V17 ja V14** — pikimad tulbad, ehk need tunnused eristavad pettusi kõige paremini.

Kuna V-tunnused on anonüümitud, ei tea me mida need täpselt mõõdavad — aga mudel oskab neid numbreid pettuse tuvastamiseks kasutada.

**Heatmap — sama info, teises vormis:**

- **Sinine** = negatiivne korrelatsioon (mida tumedam, seda tugevam seos pettusega)
- **Oranž** = positiivne korrelatsioon
- **Arv lahtris** = täpne väärtus (nt V17 = -0.33, V11 = 0.15)

Eelise eelmine tulpdiagrammi ees: numbrid on otse peal — saad täpsed väärtused ilma graafikut mõõtmata.

**Paarisdiagrammi lugemine:**

- **Diagonaal:** Jaotusgraafikud — näitavad kuidas ühe tunnuse väärtused jaotuvad. Sinine = normaalne tehing, oranž = pettus. Näed, et kaks klassi asuvad erinevates kohtades.
- **Ülejäänud lahtrid:** Iga lahter näitab kahe tunnuse seost punktpilvena — nt rida V17 ja veerg V14 näitab nende kahe tunnuse omavahelist seost.

**Mida tähele panna:** Sinised ja oranžid punktid on **selgelt eraldatud** — pettused ja normaalsed tehingud asuvad eri piirkondades. Mida selgemalt need eralduvad, seda paremini suudab mudel pettusi ära tunda. Siin on eraldus hea — eriti V17 ja V14 puhul.

## Kokkuvõte: mida andmed näitavad?

### Andmestiku üldpilt
- Andmestik on **tugevalt tasakaalustamata** — pettused moodustavad vaid **0,172%** tehingutest (492 pettust 284 807-st)
- Puuduvaid väärtusi ei ole — andmestik on puhas ja töötlemiseks valmis

### Olulised seosed

**Tehingusumma (Amount):**
- Pettuslikud tehingud on **keskmiselt väiksema summaga** kui normaalsed tehingud
- Erinevus pole aga nii suur kui esmapilgul tundub — summa üksi ei ole piisav pettuse tuvastamiseks

**Aeg (Time):**
- Normaalsed tehingud järgivad **päevarütmi** — kaks selget tippu (kaks päeva), öösel tehinguid vähe
- Absoluutarvudes tundub pettuste joon ühtlane — aga osakaaluna vaadates on teatud kellaaegadel pettusi kuni **6–7 korda rohkem**. Aeg on seega siiski kasulik signaal mudelile.

**PCA tunnused (V1–V28):**
- Kõige tugevamad korrelatsioonid klassiga on näha allolevas koodis
- Need tunnused on **kõige informatiivsemad** pettuse tuvastamisel

### Järeldus
Andmestiku suurim väljakutse on **klasside tasakaalustamatus**. Selle lahendamiseks kasutame kahte lähenemist:

> **SMOTE** — kuna pettusi on väga vähe, loob SMOTE kunstlikult uusi sarnaseid pettuse näiteid, et mudel saaks treenimise ajal piisavalt näiteid mõlemast klassist.

> **AUPRC** — tavalist täpsust (accuracy) ei saa kasutada, sest mudel saaks 99,8% "õigesti" lihtsalt öeldes kõige kohta "normaalne". AUPRC mõõdab hoopis kui hästi mudel tasakaalustab pettuste tabamist (Recall) ja valede häirete vältimist (Precision). Väärtus on 0–1, kõrgem on parem.

In [ ]:
top4 = corr_with_class.head(4).index.tolist()
neg = corr_with_class[corr_with_class < 0].head(3).index.tolist()
pos = corr_with_class[corr_with_class > 0].head(2).index.tolist()

print(f"Top 4 klassiga korreleeritud tunnust: {', '.join(top4)}")
print(f"  Negatiivne korrelatsioon: {', '.join(neg)}")
print(f"  Positiivne korrelatsioon: {', '.join(pos)}")

## Andmete eeltöötlus

### Puuduvate väärtuste käsitlemine

Enne mudeli treenimist on oluline kontrollida, kas andmestikus esineb puuduvaid väärtusi. Puuduvad väärtused võivad mudelite töö häirida ja viia ebatäpsete tulemusteni.

In [ ]:
print("Puuduvate väärtuste kontroll tehti juba andmestiku struktuuri osas.")
print("Tulemus: andmestikus puuduvaid väärtusi ei ole — edasine käsitlemine pole vajalik.")

### Kategooriliste tunnuste kodeerimine

Masinõppe mudelid saavad töötada ainult **arvudega** — teksti või kategooriaid (nt "jah/ei", "punane/sinine") nad otse ei mõista. Sellisel juhul tuleks need teisendada arvudeks.

Kontrollime, kas meie andmestikus on selliseid tunnuseid.

In [ ]:
kategoorilised = df.select_dtypes(include=['object', 'category']).columns.tolist()

if kategoorilised:
    print("Kategoorilised tunnused:", kategoorilised)
    # Kodeerimine vajadusel
    df = pd.get_dummies(df, columns=kategoorilised)
    print("Kodeerimine tehtud.")
else:
    print("Kategoorilisi tunnuseid ei leitud — kõik tunnused on arvulised, kodeerimine pole vajalik.")

### Skaleerimine (vajadusel)

Kujuta ette, et võrdled inimeste pikkust (170 cm) ja kaalu (70 kg). Need arvud on täiesti erinevatel skaaladel. Mudel võib hakata arvama, et pikkus on olulisem lihtsalt sellepärast, et arvud on suuremad.

Siin on sama probleem: `Amount` väärtused ulatuvad 0–25 000 euroni, `Time` aga 0–170 000 sekundini — need on palju suuremad kui V1–V28 tunnused.

**Skaleerimine** teisendab kõik tunnused samale skaalale (keskmine = 0), et mudel saaks neid õiglaselt võrrelda. Skaleerime ainult `Time` ja `Amount`, kuna V1–V28 on juba sobival skaalal.

> **NB:** Skaleerimine tehakse alles pärast andmete jagamist — see hoiab ära olukorra, kus testandmete info "lekib" treeningprotsessi.

In [ ]:
print("Skaleerimine teostatakse pärast train/test splitti — fit ainult treeningandmetel, et vältida andmeleket.")

## Mudeli loomine

Proovime kolme erinevat masinõppe mudelit ja võrdleme nende tulemusi:

| Mudel | Põhimõte | Lühikirjeldus |
|-------|----------|---------------|
| **Logistic Regression** | Vaatab iga tunnuse väärtust ja arvutab neile kaalud. Liidab kokku ja annab lõpptõenäosuse. | Lihtne ja kiire baasmudel — hea lähtepunkt võrdluseks |
| **Random Forest** | Ehitab palju erinevaid "otsusepuid", igaüks küsib erinevaid küsimusi. Lõpptulemus on kõigi puude hääletus. | Täpsem, kombineerib palju erinevaid vaatepunkte |
| **LightGBM** | Sarnane Random Forestiga, aga iga uus puu parandab eelmise vigu. | Kiire ja täpne, eriti suurtel andmestikel |

Kõik mudelid treenitakse SMOTE-ga tasakaalustatud andmetel.

### Andmete jagamine (Train/Test split)

Enne mudeli treenimist jagame andmed kaheks:
- **Treeningandmed (80%)** — mudel õpib nende peal
- **Testandmed (20%)** — kontrollime hiljem, kas mudel töötab ka andmetel, mida ta pole näinud

See on nagu eksamil — õpid tundidest, aga eksami küsimused on uued. Kui testida samal andmestikul millel treeniti, saaks mudel alati 100% — aga see poleks aus mõõtmine.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Skaleerimine — fit ainult treeningandmetel, transform mõlemal
scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[['Time', 'Amount']] = scaler.fit_transform(X_train[['Time', 'Amount']])
X_test[['Time', 'Amount']] = scaler.transform(X_test[['Time', 'Amount']])

print(f"Treenimisandmed: {X_train.shape[0]} rida")
print(f"Testimisandmed:  {X_test.shape[0]} rida")
print(f"\nPettuste osakaal treeningus: {y_train.mean()*100:.3f}%")
print(f"Pettuste osakaal testis:     {y_test.mean()*100:.3f}%")

### Klasside tasakaalustamatus

Meie andmestikus on ainult 0,172% pettusi. See tähendab, et kui mudel ütleks **iga** tehingu kohta "normaalne", oleks ta 99,8% ajast "õige" — aga pettusi ei tuvastaks ta üldse.

Kujuta ette, et õpetad last eristama kasse ja koeri, aga näitad talle 1000 koera pilti ja ainult 2 kassi pilti. Ta õpib kiiresti ütlema kõige kohta "koer".

**SMOTE** lahendab selle: ta loob kunstlikult uusi pettuse näiteid, et mudel näeks mõlemat klassi piisavalt palju. Seda tehakse ainult treeningandmetele — testandmed jäävad puutumata, et tulemus oleks aus.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Enne SMOTE — normaalne: {(y_train==0).sum()}, pettus: {(y_train==1).sum()}")
print(f"Pärast SMOTE — normaalne: {(y_train_sm==0).sum()}, pettus: {(y_train_sm==1).sum()}")

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=['Normaalne', 'Pettus']))
print(f"AUPRC: {average_precision_score(y_test, y_prob_lr):.4f}")

**Logistic Regression tulemuste lugemine:**

- **Normaalne tehing:** Precision 1.00 ja Recall 0.97 — mudel tuvastab normaalsed tehingud väga hästi
- **Pettus — Recall 0.92** ✓ — 92% tegelikest pettustest tabatakse
- **Pettus — Precision 0.06** ⚠️ — kui mudel ütleb "pettus", on see tegelikult pettus ainult 6% korda. 94% on valed häired — kliente blokeeritakse asjatult väga tihti.

**AUPRC 0.7249** — keskmise tasemel tulemus. See on baasmudel — Random Forest ja LightGBM peaksid andma parema tulemuse.

### LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.metrics import classification_report, average_precision_score

lgbm = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
lgbm.fit(X_train_sm, y_train_sm)
y_pred_lgbm = lgbm.predict(X_test)
y_prob_lgbm = lgbm.predict_proba(X_test)[:, 1]

print("=== LightGBM ===")
print(classification_report(y_test, y_pred_lgbm, target_names=['Normaalne', 'Pettus']))
print(f"AUPRC: {average_precision_score(y_test, y_prob_lgbm):.4f}")

**LightGBM tulemuste lugemine — palju parem kui Logistic Regression!**

- **Normaalne tehing:** Kõik näitajad 1.00 — normaalsed tehingud tuvastatakse peaaegu täiuslikult
- **Pettus — Recall 0.87** — 87% tegelikest pettustest tabatakse
- **Pettus — Precision 0.49** ✓ — kui mudel ütleb "pettus", on see õige peaaegu pooltel kordadel. Tohutu paranemine võrreldes Logistic Regression-i 6%-ga.

**AUPRC 0.8110** — märkimisväärselt parem kui Logistic Regression (0.7249). LightGBM tekitab palju vähem valesid häireid — klienti blokeeritakse ainult siis kui mudel on üsna kindel.

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=['Normaalne', 'Pettus']))
print(f"AUPRC: {average_precision_score(y_test, y_prob_rf):.4f}")

**Random Forest tulemuste lugemine — kolmest mudelist parim!**

- **Normaalne tehing:** Kõik näitajad 1.00 — normaalsed tehingud täiuslikult tuvastatud
- **Pettus — Recall 0.84** — 84% tegelikest pettustest tabatakse
- **Pettus — Precision 0.85** ✓✓ — kui mudel ütleb "pettus", on see õige 85% korda — suurepärane tulemus!

**AUPRC 0.8747** — kõrgeim kõigist kolmest mudelist.

| Mudel | Precision | Recall | AUPRC |
|-------|-----------|--------|-------|
| Logistic Regression | 0.06 | 0.92 | 0.7249 |
| LightGBM | 0.49 | 0.87 | 0.8110 |
| **Random Forest** | **0.85** | **0.84** | **0.8747** |

Random Forest on kõige tasakaalustatum — hea Precision ja Recall samaaegselt. Kliente blokeeritakse harva asjatult ja enamik pettusi tabatakse.

### SMOTE mõju võrdlus

Võrdleme Random Forest mudelit **ilma SMOTE-ta** ja **SMOTE-ga**, et näha kui palju tasakaalustamine tegelikult aitab.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

# Ilma SMOTE-ta
rf_no_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_no_smote.fit(X_train, y_train)
y_pred_ns = rf_no_smote.predict(X_test)
y_prob_ns = rf_no_smote.predict_proba(X_test)[:, 1]

# Võrdlustabel
smote_võrdlus = pd.DataFrame([
    {
        'Mudel': 'RF ilma SMOTE-ta',
        'Precision': precision_score(y_test, y_pred_ns),
        'Recall':    recall_score(y_test, y_pred_ns),
        'F1':        f1_score(y_test, y_pred_ns),
        'AUPRC':     average_precision_score(y_test, y_prob_ns),
    },
    {
        'Mudel': 'RF SMOTE-ga',
        'Precision': precision_score(y_test, y_pred_rf),
        'Recall':    recall_score(y_test, y_pred_rf),
        'F1':        f1_score(y_test, y_pred_rf),
        'AUPRC':     average_precision_score(y_test, y_prob_rf),
    },
]).set_index('Mudel').round(4)

display(smote_võrdlus.style
    .highlight_max(props='background-color: #4CAF50; color: white; font-weight: bold')
    .highlight_min(props='background-color: #f0f0f0; color: black')
)

**Kuidas tulemusi lugeda?** Roheline = parim tulemus selles veerus.

| Mõõdik | Võitja | Mida see tähendab |
|--------|--------|-------------------|
| **Precision** | Ilma SMOTE-ta | Vähem valesid häireid |
| **Recall** | SMOTE-ga | Tabab rohkem pettusi |
| **F1** | Ilma SMOTE-ta | Parem üldine tasakaal |
| **AUPRC** | SMOTE-ga | Parem peamine mõõdik |

Erinevused on väikesed, aga kuna **AUPRC on meie peamine mõõdik**, võidab SMOTE-ga mudel. SMOTE aitab eriti **Recall** puhul — mudel tabab rohkem tegelikke pettusi.

## Mudeli hindamine

Tegemist on klassifitseerimisülesandega, seega kasutame järgmisi meetrikaid:

| Meetrika | Kirjeldus |
|----------|-----------|
| **Accuracy** | Õigesti klassifitseeritud tehingute osakaal (tasakaalustamata andmetel eksitav) |
| **Precision** | Kui suur osa ennustatud pettustest on tegelikud pettused |
| **Recall** | Kui suur osa tegelikest pettustest tuvastati |
| **F1-skoor** | Precision ja Recall harmooniline keskmine |
| **AUPRC** | Ala Precision-Recall kõvera all — peamine meetrika tasakaalustamata andmetel |
| **Confusion Matrix** | Visuaalne ülevaade õigete ja valede ennustuste arvust |

### Mudelite võrdlustabel

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score

mudelid = {
    'Logistic Regression': (y_pred_lr,   y_prob_lr),
    'Random Forest':       (y_pred_rf,   y_prob_rf),
    'LightGBM':            (y_pred_lgbm, y_prob_lgbm),
}

tulemused = []
for nimi, (y_pred, y_prob) in mudelid.items():
    tulemused.append({
        'Mudel':     nimi,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'AUPRC':     average_precision_score(y_test, y_prob),
    })

tulemused_df = pd.DataFrame(tulemused).set_index('Mudel').round(4)
display(tulemused_df.style
    .highlight_max(props='background-color: #2e7d32; color: white; font-weight: bold')
    .highlight_min(props='background-color: #e0e0e0; color: #333333; font-weight: bold')
)

**Kuidas lugeda?** Tumeroheline = parim tulemus, hall = halvim.

| Mõõdik | Võitja | Mida see tähendab |
|--------|--------|-------------------|
| **Accuracy** | Random Forest | Kõige rohkem tehinguid õigesti klassifitseeritud |
| **Precision** | Random Forest | Kõige vähem valesid häireid |
| **Recall** | Logistic Regression | Tabab kõige rohkem pettusi — aga tekitab ka kõige rohkem valesid häireid |
| **F1** | Random Forest | Parim tasakaal Precision ja Recall vahel |
| **AUPRC** | Random Forest | Parim peamine mõõdik — selge võitja |

### Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (nimi, (y_pred, _)) in zip(axes, mudelid.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['Normaalne', 'Pettus'],
        cmap='Blues', ax=ax
    )
    ax.set_title(nimi)

plt.tight_layout()
plt.show()

**Confusion Matrix lugemine:**

|  | Ennustatud: Normaalne | Ennustatud: Pettus |
|--|--|--|
| **Tegelik: Normaalne** | ✅ Õigesti normaalne | ❌ Vale häire — klient blokeeriti asjatult |
| **Tegelik: Pettus** | ❌ Märkamata pettus — klient kaotas raha | ✅ Tabatud pettus |

**Kolme mudeli võrdlus:**

| | Logistic Regression | Random Forest | LightGBM |
|--|--|--|--|
| ✅ Õigesti normaalne | 55 403 | 56 849 | 56 776 |
| ❌ Vale häire | 1 461 | **15** | 88 |
| ❌ Märkamata pettus | 8 | 16 | 13 |
| ✅ Tabatud pettus | 90 | 82 | 85 |

- **Logistic Regression** tekitab 1461 vale häiret — liiga palju asjatuid blokkimisi
- **Random Forest** ainult 15 vale häiret ✓ — kuid laseb 16 pettust läbi
- **LightGBM** on keskel — 88 vale häiret, 13 märkamata pettust

**Random Forest** on kõige täpsem valesid häireid vältides.

### Precision-Recall kõver

In [ ]:
from sklearn.metrics import precision_recall_curve

plt.figure(figsize=(10, 6))

for nimi, (_, y_prob) in mudelid.items():
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)
    plt.plot(recall, precision, label=f"{nimi} (AUPRC={auprc:.4f})")

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall kõver')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**Precision-Recall kõvera lugemine:**

- **X-telg (Recall)** — kui suur osa pettustest tabatakse (0 = mitte ühtegi, 1 = kõik)
- **Y-telg (Precision)** — kui täpne mudel on (1 = ei tee ühtegi vale häiret, 0 = ainult valed häired)

**Ideaalne mudel** püsiks kogu aeg ülemises paremas nurgas — tabaks kõik pettused ilma ühegi vale häireta. Päriselus tuleb valida kompromiss: rohkem pettusi tabada tähendab ka rohkem valesid häireid.

**Mida kõrgemal kõver püsib, seda parem mudel:**
- **Oranž (Random Forest, AUPRC=0.8747)** — püsib kõige kõrgemal kauem → parim
- **Roheline (LightGBM, AUPRC=0.8110)** — hea kuni ~80% recall-ini, siis langeb
- **Sinine (Logistic Regression, AUPRC=0.7249)** — langeb kõige kiiremini → nõrgim

**AUPRC** = ala kõvera all. Mida suurem ala, seda parem mudel.

### Tulemuste selgitus

#### Mida numbrid tähendavad?

**Precision** — kui mudel ütleb "see on pettus", kui tihti ta on õige?
- Näide: Precision 0.90 tähendab, et 90% korda kui mudel märgib pettuse, on see tõesti pettus. 10% on valed häired (klient blokeeritakse asjatult).

**Recall** — kui paljud tegelikud pettused mudel tabab?
- Näide: Recall 0.80 tähendab, et 80% tegelikest pettustest tuvastatakse. 20% pettustest "libiseb läbi".

**Kumb on olulisem?**
Sõltub ärilisest eesmärgist:
- Kui **klient ei tohi kahju saada** → Recall on tähtsam (taba kõik pettused, isegi kui mõni normaalne tehing blokeeritakse)
- Kui **klient ei tohi tülikalt blokeerida** → Precision on tähtsam

**F1-skoor** — tasakaalustab mõlemad. Hea üldine mõõdik.

**AUPRC** — parim mõõdik tasakaalustamata andmetel. Väärtus vahemikus 0–1, kõrgem on parem. Juhuslik mudel saaks ~0.002 (pettuste osakaal).

#### Confusion Matrix lugemine

|  | Ennustatud: Normaalne | Ennustatud: Pettus |
|--|--|--|
| **Tegelik: Normaalne** | ✅ Õige (TN) | ❌ Vale häire (FP) |
| **Tegelik: Pettus** | ❌ Märkamata pettus (FN) | ✅ Tabatud pettus (TP) |

Pettuste tuvastamisel on **FN (märkamata pettus) kõige kulukam** viga.

### Tunnuste olulisus (Feature Importance)

Vaatame, millised tunnused on Random Forest ja LightGBM mudelite jaoks kõige olulisemad pettuse tuvastamisel.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (mudel, nimi) in zip(axes, [(rf, 'Random Forest'), (lgbm, 'LightGBM')]):
    importances = pd.Series(mudel.feature_importances_, index=X_train.columns)
    importances.nlargest(15).sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'{nimi} — Top 15 tunnust')
    ax.set_xlabel('Olulisus')

plt.tight_layout()
plt.show()

**Feature Importance lugemine — mida pikem tulp, seda olulisem tunnus:**

**Random Forest (vasak):**
- **V14** on selgelt kõige olulisem — palju pikem tulp kui teistel
- **V10, V4, V12, V17** on samuti olulised

**LightGBM (parem):**
- **V4** on kõige olulisem, järgneb **V14**
- **Time ja Amount** on LightGBM jaoks olulised — Random Forestil neid top 15-s pole

**Mõlemad mudelid nõustuvad:** V14, V4, V12, V17, V3 esinevad mõlemas nimekirjas — need on tõeliselt informatiivsed tunnused pettuse tuvastamisel.

> Numbrite skaalad on erinevad (RF: 0–0.2, LightGBM: 0–200) — oluline on suhteline järjestus, mitte arv ise.

### Läve optimeerimine

Mudel annab igale tehingule **tõenäosuse** vahemikus 0–1 (nt 0.73 = 73% tõenäosus, et tehing on pettus). Vaikimisi kasutame läve **0.5** — kui tõenäosus on üle 0.5, märgime tehingu pettuseks.

Aga kas 0.5 on parim valik?
- **Madalam lävi** (nt 0.3) → tabame rohkem pettusi, aga blokeerime ka rohkem normaalseid tehinguid
- **Kõrgem lävi** (nt 0.7) → vähem valesid häireid, aga mõni pettus libiseb läbi

Allpool otsime automaatselt parima läve, mis annab kõrgeima F1-skoori.

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

laevad_optimaalsed = {}

for ax, (nimi, (_, y_prob)) in zip(axes, mudelid.items()):
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_test, y_prob)
    f1_arr = 2 * (precision_arr[:-1] * recall_arr[:-1]) / (precision_arr[:-1] + recall_arr[:-1] + 1e-9)
    opt_idx = np.argmax(f1_arr)
    opt_threshold = thresholds[opt_idx]
    laevad_optimaalsed[nimi] = opt_threshold

    ax.plot(thresholds, precision_arr[:-1], label='Precision')
    ax.plot(thresholds, recall_arr[:-1], label='Recall')
    ax.plot(thresholds, f1_arr, label='F1', linestyle='--')
    ax.axvline(opt_threshold, color='red', linestyle=':', label=f'Optimaalne lävi={opt_threshold:.2f}')
    ax.set_title(nimi)
    ax.set_xlabel('Lävi')
    ax.legend()

plt.tight_layout()
plt.show()

print("\nOptimaalse läve tulemused (F1 järgi):")
for nimi, laev in laevad_optimaalsed.items():
    y_prob = mudelid[nimi][1]
    y_pred_opt = (y_prob >= laev).astype(int)
    print(f"\n{nimi} (lävi={laev:.2f}):")
    print(classification_report(y_test, y_pred_opt, target_names=['Normaalne', 'Pettus']))

**Läve optimeerimise graafiku lugemine:**

Iga graafik näitab, mis juhtub kui muudame otsustusläve (0 = kõik tehingud märgitakse pettuseks, 1 = mitte miski pole pettus). Punane joon näitab, kus F1 on maksimaalne — see on optimaalne lävi.

**Kolm joont:**
- **Sinine (Precision)** — mida kõrgem lävi, seda täpsemad ennustused (vähem valesid häireid)
- **Oranž (Recall)** — mida kõrgem lävi, seda vähem pettusi tabatakse
- **Roheline katkendlik (F1)** — tasakaal nende vahel

**Kolme mudeli võrdlus:**
- **Logistic Regression** (lävi 1.00) — mudel on nii ebakindel, et F1 on parim alles maksimaalse kindlusega. Problemaatiline.
- **Random Forest** (lävi 0.77) — kõige mõistlikum. Sinised ja oranžid jooned ristuvad kenasti, F1 saavutab hea tasakaalu.
- **LightGBM** (lävi 0.99) — sarnaselt LR-iga vajab väga kõrget kindlust enne pettuse märkimist.

**Järeldus:** Random Forest on kõige stabiilsem — lävi 0.77 näitab, et mudel on oma ennustustes enesekindel.

### Äriline läve optimeerimine (Cost-Sensitive Threshold)

Eelmine läve optimeerimine kasutas F1-skoori — see kohtleb FP ja FN võrdselt. Reaalses panganduses on need **erineva hinnaga**:

| Viga | Tähendus | Hinnanguline kulu |
|------|----------|-------------------|
| **FN** (märkamata pettus) | Klient kaotab raha, pank hüvitab | ~tehingu summa (keskm. ~122€) |
| **FP** (vale häire) | Normaalne tehing blokeeritakse | ~10€ (klienditeenindus, ebamugavus) |

Leiame läve, mis **minimeerib kogukulu** antud kulukaalude juures.

In [ ]:
KULU_FN = 122  # märkamata pettuse kulu (€)
KULU_FP = 10   # vale häire kulu (€)

fig, axes = plt.subplots(1, len(mudelid), figsize=(18, 5))

print("Ärilise kulufunktsiooni tulemused:\n")

for ax, (nimi, (_, y_prob)) in zip(axes, mudelid.items()):
    thresholds = np.linspace(0.01, 0.99, 200)
    kulud = []

    for t in thresholds:
        y_pred_t = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
        kogukulu = fn * KULU_FN + fp * KULU_FP
        kulud.append(kogukulu)

    opt_idx  = np.argmin(kulud)
    opt_laev = thresholds[opt_idx]
    min_kulu = kulud[opt_idx]

    ax.plot(thresholds, kulud, color='steelblue')
    ax.axvline(opt_laev, color='red', linestyle='--',
               label=f'Optimaalne lävi={opt_laev:.2f}\nKulu={min_kulu:.0f}€')
    ax.set_title(nimi)
    ax.set_xlabel('Lävi')
    ax.set_ylabel('Kogukulu (€)')
    ax.legend()

    # Võrdlus vaikimisi lävega 0.5
    y_pred_05 = (y_prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_05).ravel()
    kulu_05 = fn * KULU_FN + fp * KULU_FP

    print(f"{nimi}:")
    print(f"  Vaikimisi lävi 0.50 → kogukulu: {kulu_05:.0f}€")
    print(f"  Optimaalne lävi {opt_laev:.2f} → kogukulu: {min_kulu:.0f}€")
    print(f"  Kokkuhoid: {kulu_05 - min_kulu:.0f}€\n")

plt.tight_layout()
plt.show()

**Ärilise kulufunktsiooni lugemine:**

Kulu arvutatakse päris eurodes, kasutades kahte hinnangulist väärtust:
- **122€ märkamata pettuse eest** — andmestiku keskmine tehingusumma (kui pettus jääb märkamata, kaotab klient keskmiselt selle summa)
- **10€ vale häire eest** — hinnanguline kulu kliendi asjatule blokeerimisele (klienditeenindus, ebamugavus)

> **NB:** Need on hinnangulised väärtused. Reaalses rakenduses peaks pank ise määrama täpsed kulud.

**Kogukulu = märkamata pettused × 122€ + valed häired × 10€**

Graafikul näitab kõver kogukulu erinevatel läveväärtustel — punane joon on kõige odavam punkt.

| Mudel | Vaikimisi lävi 0.5 | Optimaalne lävi | Kokkuhoid |
|-------|-------------------|-----------------|-----------|
| Logistic Regression | 15 586€ | 2 154€ (lävi 0.99) | **13 432€** |
| Random Forest | 2 102€ | 1 490€ (lävi 0.34) | **612€** |
| LightGBM | 2 466€ | 2 016€ (lävi 0.73) | **450€** |

**Random Forest on kõige odavam** — isegi vaikimisi lävega ainult 2 102€. Logistic Regression on vaikimisi lävega katastroofiliselt kallis (15 586€) — 1 461 vale häiret on kulukas.

## Tulemuste tõlgendamine

### Mida mudel õppis?

In [ ]:
parim_mudel = tulemused_df['AUPRC'].idxmax()
parim_auprc = tulemused_df['AUPRC'].max()
parim_recall = tulemused_df.loc[parim_mudel, 'Recall']
parim_precision = tulemused_df.loc[parim_mudel, 'Precision']

print("=== Mida mudel õppis? ===\n")
print(f"Parim mudel AUPRC järgi: {parim_mudel} ({parim_auprc:.4f})")
print(f"  → Tabab {parim_recall*100:.1f}% tegelikest pettustest (Recall)")
print(f"  → {parim_precision*100:.1f}% pettuseks märgitud tehingutest on tõesti pettused (Precision)")
print()
print("Mida mudelid õppisid:")
print("  • Pettuslikud tehingud erinevad normaalsetest kindlate PCA-tunnuste (V14, V17 jne) mustrite poolest")
print("  • Pettused esinevad sagedamini teatud kellaaegadel ja väiksemate summadega")
print("  • Mudel suudab tuvastada statistilisi anomaaliaid, mida inimsilm ei märkaks")

### Millised tunnused olid olulised?

In [ ]:
top_rf   = pd.Series(rf.feature_importances_,   index=X_train.columns).nlargest(5)
top_lgbm = pd.Series(lgbm.feature_importances_, index=X_train.columns).nlargest(5)

print("Top 5 olulisemat tunnust:")
print(f"\nRandom Forest:  {', '.join(top_rf.index.tolist())}")
print(f"LightGBM:       {', '.join(top_lgbm.index.tolist())}")

yhised = set(top_rf.index) & set(top_lgbm.index)
print(f"\nMõlemas mudelis olulised tunnused: {', '.join(yhised)}")
print("\nTõlgendus:")
print("  • Need tunnused on PCA komponendid — nende tegelik tähendus on konfidentsiaalne")
print("  • Kuid mõlemad mudelid nõustuvad, et samad tunnused on pettuse tuvastamisel kõige informatiivsemad")
print("  • See kinnitab, et tulemused ei ole juhuslikud — mudelid leiavad järjepidevalt samu mustreid")

### Kas tulemused on usaldusväärsed?

**Miks jah:**
- Testimine tehti **andmetel, mida mudel treenimise ajal ei näinud** — tulemus on aus
- Andmete jagamisel jälgisime, et pettuste osakaal oleks treeningu- ja testandmetes võrdne
- SMOTE rakendati ainult treeningandmetele — testandmed on originaalsed ja moonutamata
- Mitu erinevat mudelit jõuavad **sarnastele järeldustele** — see suurendab usaldust
- Kasutasime AUPRC-d, mis sobib tasakaalustamata andmetele paremini kui lihtsalt täpsus

**Piirangud — mida tuleks meeles pidada:**
- Andmestik pärineb **2013. aastast** — pettuste mustrid võivad olla tänaseks muutunud
- V1–V28 on anonüümitud — me ei tea, mida need tegelikult mõõdavad
- Andmed on ainult **Euroopa tehingutest** — ei pruugi sobida teistele piirkondadele
- Ainult **2 päeva andmeid** — pikemaajalised mustrid (nt hooajalisus) pole kaetud

## Kokkuvõte

### Peamised järeldused

In [ ]:
parim_mudel  = tulemused_df['AUPRC'].idxmax()
parim_auprc  = tulemused_df['AUPRC'].max()
parim_f1     = tulemused_df.loc[parim_mudel, 'F1']
parim_recall = tulemused_df.loc[parim_mudel, 'Recall']

print("=" * 55)
print("PEAMISED JÄRELDUSED")
print("=" * 55)
print(f"""
1. ANDMESTIK
   • 284 807 tehingut, millest 492 (0.172%) on pettused
   • Andmestik on tugevalt tasakaalustamata — see on
     peamine tehniline väljakutse

2. PARIM MUDEL
   • {parim_mudel} saavutas kõrgeima AUPRC: {parim_auprc:.4f}
   • F1-skoor: {parim_f1:.4f} | Recall: {parim_recall:.4f}
   • Mudel tabab enamiku pettustest säilitades
     mõistliku arvu valesid häireid

3. OLULISEMAD TUNNUSED
   • PCA tunnused V14, V17, V12 ja V10 on kõige
     informatiivsemad pettuse tuvastamisel
   • Amount ja Time annavad lisainfot, kuid on
     vähem olulised kui V-tunnused

4. EELTÖÖTLUS
   • SMOTE parandas märkimisväärselt mudeli võimet
     tuvastada pettusi tasakaalustamata andmetel
   • Läve optimeerimine võimaldab recall ja precision
     vahelist tasakaalu ärivajaduse järgi kohandada
""")

### Mida teeksid järgmise sammuna paremini?

| Samm | Kirjeldus |
|------|-----------|
| **Parameetrite häälestamine** | Iga mudel on nagu raadiosaatja — sellel on nupud, mida saab keerata. Praegu kasutame vaikeasendeid. Neid nuppe optimeerides saaks täpsust parandada. |
| **Ristvalideerimine** | Praegu jagasime andmed üks kord kaheks (treening/test). Usaldusväärsema tulemuse saaks, kui jagada andmed mitu korda erinevalt ja võtta keskmine tulemus. |
| **Anomaaliate tuvastamine** | Praegu õpetasime mudelile "näited pettustest". Alternatiiv — lasta mudelil ise otsida "imelikke" tehinguid ilma et ta teaks mis on pettus. |
| **Ajapõhine testimine** | Päriselus treenitakse mudel vanadel andmetel ja kasutatakse uutel. Meil on 2 päeva andmeid — ideaalis testida esimese päeva peal treenimine ja teise päeva peal testimine. |
| **Täpsemad kulud** | Meie 122€ ja 10€ olid hinnangud. Pangaga koostöös saaks määrata täpsemad kulud — see muudaks läve valiku palju täpsemaks. |
| **Mudeli seletavus** | Praegu mudel ütleb "see on pettus" — aga miks? Lisatööriistad saaksid selgitada iga otsust eraldi: "see tehing on kahtlane, sest V14 on ebatavaliselt madal". |

## Raport (PDF)

In [ ]:
# ── PDF RAPORT ─────────────────────────────────────────────────────────
# Jooksuta alles PÄRAST kõigi eelnevate lahtrite täitmist!
# Vajalikud muutujad: df, corr_with_class, tulemused_df, rf, lgbm, X_test, y_test
# Tulemus salvestatakse faili: raport.pdf
# ─────────────────────────────────────────────────────────────────────────

from fpdf import FPDF
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import os

class Raport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 12)
        self.set_text_color(30, 30, 30)
        self.cell(0, 10, 'Krediitkaardi pettuste tuvastamine', align='C', new_x='LMARGIN', new_y='NEXT')
        self.ln(2)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.set_text_color(128)
        self.cell(0, 10, f'Lehekülg {self.page_no()}', align='C')

pdf = Raport()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# ── 1. Probleemi kirjeldus ────────────────────────────────────────────────
pdf.set_font('Helvetica', 'B', 13)
pdf.set_text_color(0, 70, 127)
pdf.cell(0, 10, '1. Probleemi kirjeldus', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', size=10)
pdf.set_text_color(30, 30, 30)
pdf.multi_cell(0, 7, (
    "Eesmark: tuvastada pettuslikud krediitkaardi tehingud automaatselt.\n"
    "Andmestik: 284 807 tehingut (septembrist 2013), millest 492 (0.172%) on pettused.\n"
    "Andmestik on tugevalt tasakaalustamata - peamine tehniline valjakutse.\n"
    "Tunnused V1-V28 on anon\u00fcmitud (PCA), Time ja Amount on originaalsed."
))
pdf.ln(3)

# ── 2. Andmestiku graafikud ───────────────────────────────────────────────
pdf.set_font('Helvetica', 'B', 13)
pdf.set_text_color(0, 70, 127)
pdf.cell(0, 10, '2. Andmestiku graafikud', new_x='LMARGIN', new_y='NEXT')

fig, ax = plt.subplots(figsize=(5, 3))
df['Class'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Klasside jaotus')
ax.set_xlabel('Klass (0=normaalne, 1=pettus)')
ax.set_ylabel('Arv')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('_g1.png', dpi=100)
plt.close()

fig, ax = plt.subplots(figsize=(6, 3))
corr_with_class.head(10).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 10 tunnuse korrelatsioon klassiga')
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('_g2.png', dpi=100)
plt.close()

pdf.set_font('Helvetica', 'I', 9)
pdf.set_text_color(80, 80, 80)
pdf.cell(0, 6, 'Joonis 1: Klasside jaotus (vasak) ja korrelatsioon klassiga (parem)', new_x='LMARGIN', new_y='NEXT')
pdf.image('_g1.png', w=90)
y_pos = pdf.get_y()
pdf.set_xy(105, y_pos - 55)
pdf.image('_g2.png', w=95)
pdf.ln(5)

# ── 3. Mudelite tulemuste tabel ───────────────────────────────────────────
pdf.set_font('Helvetica', 'B', 13)
pdf.set_text_color(0, 70, 127)
pdf.cell(0, 10, '3. Mudelite tulemuste tabel', new_x='LMARGIN', new_y='NEXT')

headers = ['Mudel', 'Precision', 'Recall', 'F1', 'AUPRC']
col_w = [60, 30, 30, 30, 30]

pdf.set_font('Helvetica', 'B', 10)
pdf.set_fill_color(0, 70, 127)
pdf.set_text_color(255, 255, 255)
for h, w in zip(headers, col_w):
    pdf.cell(w, 8, h, border=1, fill=True, align='C')
pdf.ln()

model_names = ['Logistic Regression', 'Random Forest', 'LightGBM']
colors = [(240,248,255), (255,255,255), (240,248,255)]
pdf.set_text_color(30, 30, 30)
parim = tulemused_df['AUPRC'].idxmax()
for name, bg in zip(model_names, colors):
    row = tulemused_df.loc[name]
    if name == parim:
        pdf.set_fill_color(200, 230, 200)
    else:
        pdf.set_fill_color(*bg)
    pdf.set_font('Helvetica', 'B' if name == parim else '', 10)
    label = name + (' *' if name == parim else '')
    pdf.cell(col_w[0], 8, label, border=1, fill=True)
    pdf.set_font('Helvetica', size=10)
    for val, w in zip([row['Precision'], row['Recall'], row['F1'], row['AUPRC']], col_w[1:]):
        pdf.cell(w, 8, f'{val:.4f}', border=1, fill=True, align='C')
    pdf.ln()
pdf.ln(2)

pdf.set_font('Helvetica', 'I', 9)
pdf.set_text_color(80, 80, 80)
pdf.multi_cell(0, 6, '* = parim mudel (AUPRC jargi). Precision = mitu % pettushaire test on oiged. Recall = mitu % pettustest tabati. AUPRC = uldine hindamisnaitaja tasakaalustamata andmetel.')
pdf.ln(3)

# ── 4. Confusion Matrix (parim mudel) ────────────────────────────────────
pdf.add_page()
pdf.set_font('Helvetica', 'B', 13)
pdf.set_text_color(0, 70, 127)
pdf.cell(0, 10, f'4. Confusion Matrix - {parim}', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', size=10)
pdf.set_text_color(30, 30, 30)
pdf.multi_cell(0, 6, (
    "Confusion matrix naitab neli tulemust:\n"
    "  Vasak ulanurk (TN): mudel utles 'normaalne' ja oli oige\n"
    "  Parem ulanurk (FP): mudel utles 'pettus' aga oli vale (vale haire)\n"
    "  Vasak alanurk (FN): mudel utles 'normaalne' aga tegelikult oli pettus (vahele jainud!)\n"
    "  Parem alanurk (TP): mudel utles 'pettus' ja oli oige (tabatud pettus)\n"
))
pdf.ln(2)

mudeli_obj = {'Random Forest': rf, 'LightGBM': lgbm}
y_pred_best = mudeli_obj[parim].predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normaalne', 'Pettus'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix: {parim}')
plt.tight_layout()
plt.savefig('_g3.png', dpi=100)
plt.close()

pdf.image('_g3.png', x=15, w=85)
x_text = 108
y_cm = pdf.get_y() - 72
pdf.set_xy(x_text, y_cm)
pdf.set_font('Helvetica', size=10)
pdf.set_text_color(30, 30, 30)
pdf.multi_cell(90, 7, (
    f"Tulemused arvudes:\n\n"
    f"  Oige normaalne (TN): {tn:,}\n"
    f"  Vale haire (FP): {fp}\n"
    f"  Vahele jainud pettus (FN): {fn}\n"
    f"  Tabatud pettus (TP): {tp}\n\n"
    f"Recall: {tp}/{tp+fn} = {tp/(tp+fn)*100:.1f}% pettustest tabati\n"
    f"Precision: {tp}/{tp+fp} = {tp/(tp+fp)*100:.1f}% hairetest oli oige"
))
pdf.ln(5)

# ── 5. Peamised järeldused ────────────────────────────────────────────────
pdf.set_font('Helvetica', 'B', 13)
pdf.set_text_color(0, 70, 127)
pdf.cell(0, 10, '5. Peamised jareldused', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', size=10)
pdf.set_text_color(30, 30, 30)

auprc_val = tulemused_df['AUPRC'].max()
recall_val = tulemused_df.loc[parim, 'Recall']
precision_val = tulemused_df.loc[parim, 'Precision']

pdf.multi_cell(0, 7, (
    f"Parim mudel: {parim} (AUPRC={auprc_val:.4f})\n"
    f"  - Tabab {recall_val*100:.1f}% tegelikest pettustest\n"
    f"  - {precision_val*100:.1f}% pettuseks margitud tehingutest on tegelikult pettused\n\n"
    "Olulisemad tunnused: V14, V4, V12, V17 (molemal mudelil yhised)\n\n"
    "Suurim valjakutse: klasside tasakaalustamatus (0.172% pettusi)\n"
    "Lahendus: SMOTE + AUPRC meetrika (mitte tavaline tapsus)\n\n"
    "Ariline moju: Random Forest optimaalse lavega saastab ~612 eurot\n"
    "vorreldus vaikimisi lavega (0.5)."
))

# ── Salvesta ja koristus ──────────────────────────────────────────────────
output = 'c:/Users/Administrator/creditcard/raport.pdf'
pdf.output(output)

for f in ['_g1.png', '_g2.png', '_g3.png']:
    if os.path.exists(f): os.remove(f)

print(f'Raport salvestatud: {output}')
